In [ ]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import os

# ---------------------------------------------------------------------------
# Configuracion externalizada (widgets de Databricks / variables de entorno).
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")


def _cfg(name: str, default: str = "") -> str:
    """Resuelve un parametro: widget -> variable de entorno -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog", "job_offers")
SILVER_SCHEMA = _cfg("silver_schema", "silver")
GOLD_SCHEMA = _cfg("gold_schema", "gold")


def silver_table(name: str) -> str:
    return f"{CATALOG}.{SILVER_SCHEMA}.{name}"


In [ ]:
df_linkedin=spark.read.table(silver_table("offers_linkedin"))
df_infojobs=spark.read.table(silver_table("offers_infojobs"))
df_indeed=spark.read.table(silver_table("offers_indeed"))
df_multisite=spark.read.table(silver_table("offers_multi_site"))

In [ ]:
display(df_linkedin)

In [ ]:
from functools import reduce
from pyspark.sql import DataFrame

# Orden de columnas puede variar, pero nombres iguales
result = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    [df_linkedin, df_infojobs, df_indeed, df_multisite])

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

UNICODE_DASHES = r"[\u2010\u2011\u2012\u2013\u2014\u2212\uFE58\uFE63\uFF0D]"

def _norm(c):
    c = F.trim(F.lower(c))
    c = F.regexp_replace(c, UNICODE_DASHES, "-")
    c = F.regexp_replace(c, r"\s*-\s*", "-")
    return F.trim(F.regexp_replace(c, r"\s+", " "))

def _keep_best(df, partition_cols):
    w = Window.partitionBy(*partition_cols).orderBy(
        F.when(F.col("description_clean").isNotNull() & (F.trim(F.col("description_clean")) != ""), 1).otherwise(0).desc(),
        F.when(F.col("salary_min_annual").isNotNull(), 1).otherwise(0).desc(),
        F.col("posted_date").desc_nulls_last())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))

result = (result
          .withColumn("title_norm", _norm(F.col("title")))
          .withColumn("company_norm", _norm(F.col("company_name"))))

result = _keep_best(result, ["job_id"])

w_fill = Window.partitionBy("title_norm", "company_norm")
result = result.withColumn(
    "salary_filled",
    F.coalesce(F.col("salary_min_annual"),
               F.first(F.col("salary_min_annual"), ignorenulls=True).over(w_fill)))

result = _keep_best(result, ["title_norm", "company_norm", "salary_filled"])
result = result.drop("title_norm", "company_norm", "salary_filled")

In [ ]:
import Prepare_Gold as dg

gold_tables = dg.build_gold(spark, result)

dg.write_gold(gold_tables, catalog=CATALOG, schema=GOLD_SCHEMA)



Dim_Companies

In [ ]:
companies_df_linkedin=spark.read.table(silver_table("companies_linkedin"))
companies_df_indeed=spark.read.table(silver_table("companies_indeed"))
companies_df_infojobs=spark.read.table(silver_table("companies_infojobs"))
companies_df_multi_site=spark.read.table(silver_table("companies_multi_site"))

In [ ]:
from pyspark.sql import functions as F

companies_df_linkedin = companies_df_linkedin.select(
    F.trim("company_name").alias("company_name"),
    F.col("company_industry"),
    F.col("company_description"),
    F.col("company_size"),
    F.lit("linkedin").alias("source"),
)

companies_df_indeed = companies_df_indeed.select(
    F.trim("company_name").alias("company_name"),      # Indeed usa 'company'
    F.col("company_rating"),
    F.col("company_review_count"),
    F.lit("indeed").alias("source"),
)
#2. Union (las columnas que falten en un lado se rellenan con NULL)
companies_df = companies_df_linkedin.unionByName(companies_df_indeed, allowMissingColumns=True)
#3. Añadir los 2 dataframes que solo tienen company_name
only_names = companies_df_multi_site.select(F.col("company_name")) \
    .union(companies_df_infojobs.select(F.col("company_name")))
#Al unir con allowMissingColumns=True, las columnas extra de companies quedan a NULL para estas filas:
companies_final = companies_df.unionByName(only_names, allowMissingColumns=True)
#4. Normalizar y deduplicar (imprescindible: la misma empresa aparece en varias fuentes)
from pyspark.sql import Window

w = Window.partitionBy("_key").orderBy(
    F.when(F.col("company_industry").isNotNull(), 1).otherwise(0).desc(),
    F.when(F.col("company_size").isNotNull(), 1).otherwise(0).desc(),
    F.when(F.col("company_rating").isNotNull(), 1).otherwise(0).desc(),
    F.col("source").asc(),
)
companies_final = (
    companies_final
    .withColumn("_key", F.lower(F.trim(F.regexp_replace("company_name", r"\s+", " "))))
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_key")
)
companies_final = companies_final.filter(
    ~(F.col("company_name").rlike(r"^\d+[,.]\d{1,2}$"))
    & (F.trim(F.col("company_name")) != "")
)

In [ ]:
companies_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.companies_complete_catalog")

In [ ]:
# ---------------------------------------------------------------------------
# Diagnostico de rendimiento (spec 010, T-04/T-05): layout y deletion vectors
# de las cuatro silvers. Solo lee metadatos y el _delta_log; imprime una linea
# con la marca PERF_DIAGNOSTICS= y nunca lanza una excepcion.
# ---------------------------------------------------------------------------
import json

from pyspark.sql import functions as F

DIAGNOSTICS_TABLES = (
    "offers_indeed",
    "offers_infojobs",
    "offers_linkedin",
    "offers_multi_site",
)


def _action_fields(frame, action):
    """Campos de un struct del log; lista vacia si la accion no aparece."""
    if action not in frame.columns:
        return []
    data_type = frame.schema[action].dataType
    return list(data_type.fieldNames()) if hasattr(data_type, "fieldNames") else []


def _delta_log_state(location):
    """Estado del _delta_log: ficheros actuales y ficheros con deletion vector."""
    log_path = str(location).rstrip("/") + "/_delta_log"
    entries = dbutils.fs.ls(log_path)
    commits = sorted(
        entry.path for entry in entries if entry.name.endswith(".json")
    )
    checkpoint = any(".checkpoint." in entry.name for entry in entries)
    note = (
        "Recuento desde los commits JSON: con checkpoint o ficheros compactados "
        "puede ser un limite inferior; los compactados se leen con su version "
        "inicial. withDv cuenta ficheros del estado con deletion vector, no "
        "filas borradas."
    )
    if not commits:
        return {
            "files": 0,
            "withDv": 0,
            "checkpoint": checkpoint,
            "note": "No hay commits JSON en el _delta_log. " + note,
        }
    log = spark.read.json(commits).withColumn(
        "version",
        F.regexp_extract(
            F.element_at(F.split(F.input_file_name(), "/"), -1),
            r"^(\d+)",
            1,
        ).cast("int"),
    )
    file_columns = [
        F.col(f"{action}.path")
        for action in ("add", "remove")
        if "path" in _action_fields(log, action)
    ]
    if not file_columns:
        return {
            "files": 0,
            "withDv": 0,
            "checkpoint": checkpoint,
            "note": "El log no tiene acciones add/remove con path. " + note,
        }
    file_column = (
        file_columns[0]
        if len(file_columns) == 1
        else F.coalesce(*file_columns)
    )
    state = log.withColumn("file", file_column).where(
        F.col("file").isNotNull()
    )
    # Ultima accion por fichero: version y desempate a favor de add.
    if "path" in _action_fields(state, "add"):
        state = state.withColumn(
            "is_add", F.when(F.col("add.path").isNotNull(), 1).otherwise(0)
        )
    else:
        state = state.withColumn("is_add", F.lit(0))
    state = state.withColumn(
        "order_key", F.col("version") * 2 + F.col("is_add")
    )
    has_dv = "deletionVector" in _action_fields(state, "add")
    if has_dv:
        last = state.groupBy("file").agg(
            F.max_by(
                F.struct(
                    F.col("add.deletionVector").alias("dv"),
                    F.col("is_add").alias("is_add"),
                ),
                F.col("order_key"),
            ).alias("last")
        )
        current = last.where(F.col("last.is_add") == 1)
        files = current.count()
        with_dv = current.where(F.col("last.dv").isNotNull()).count()
    else:
        last = state.groupBy("file").agg(
            F.max_by(F.col("is_add"), F.col("order_key")).alias("is_add")
        )
        files = last.where(F.col("is_add") == 1).count()
        with_dv = 0
    return {
        "files": files,
        "withDv": with_dv,
        "checkpoint": checkpoint,
        "note": note,
    }


def _table_layout(name):
    """numFiles, sizeInBytes y location de una silver, mas el estado del log."""
    row = spark.sql(f"DESCRIBE DETAIL {silver_table(name)}").first()
    detail = row.asDict() if row is not None else {}
    num_files = detail.get("numFiles")
    info = {
        "numFiles": num_files,
        "sizeInBytes": detail.get("sizeInBytes"),
        "location": detail.get("location"),
    }
    location = info["location"]
    if not location:
        info.update(
            {
                "files": 0,
                "withDv": 0,
                "checkpoint": False,
                "note": "DESCRIBE DETAIL no devolvio location.",
            }
        )
        return info
    try:
        info.update(_delta_log_state(location))
    except Exception as exc:  # el layout ya leido no se pierde
        info["log_error"] = str(exc)[:300]
        return info
    if num_files is not None and info["files"] != int(num_files):
        info["mismatch"] = True
        info["mismatch_note"] = (
            "El recuento del _delta_log ({}) no coincide con numFiles de "
            "DESCRIBE DETAIL ({}); revisa checkpoint y compactacion."
        ).format(info["files"], int(num_files))
    else:
        info["mismatch"] = False
    return info


payload = {}
for _name in DIAGNOSTICS_TABLES:
    try:
        payload[_name] = _table_layout(_name)
    except Exception as exc:  # el diagnostico nunca rompe el job
        payload[_name] = {"error": str(exc)[:300]}

print("PERF_DIAGNOSTICS=" + json.dumps(payload, sort_keys=True, default=str))

In [ ]:
# ---------------------------------------------------------------------------
# Diagnostico de rendimiento (spec 010, T-04): proxy del join que decide el
# ON del MERGE de offers_linkedin. Imprime una linea con la marca
# PERF_MERGE_PLAN= y nunca lanza una excepcion.
# ---------------------------------------------------------------------------
import json

PERF_MERGE_T_VIEW = "__perf_merge_t"
PERF_MERGE_S_VIEW = "__perf_merge_s"
NOTE = (
    "El plan interno del MERGE no es accesible sin ejecutarlo: EXPLAIN MERGE "
    "solo devuelve Execute MergeIntoCommand con los hijos logicos. Esta celda "
    "mide un PROXY del join del ON (vistas vacias, FULL OUTER JOIN); el MERGE "
    "real usa inner y left outer con datos reales, asi que el Exchange y los "
    "joins del proxy son ORIENTATIVOS y no demuestran el plan del MERGE ni la "
    "poda por fichero. Con datos reales se mediran en el run."
)
payload = {"scope": "proxy_full_outer_join_on_key", "note": NOTE}

try:
    import Silver_Incremental as inc

    target = silver_table("offers_linkedin")
    payload["target"] = target
    spark.sql(
        f"SELECT * FROM {target} LIMIT 0"
    ).createOrReplaceTempView(PERF_MERGE_T_VIEW)
    spark.sql(
        f"SELECT * FROM {target} LIMIT 0"
    ).createOrReplaceTempView(PERF_MERGE_S_VIEW)
    explain_sql = (
        f"EXPLAIN SELECT count(*) FROM {PERF_MERGE_T_VIEW} AS t "
        f"FULL OUTER JOIN {PERF_MERGE_S_VIEW} AS s "
        f"ON {inc.key_sql('t')} = {inc.key_sql('s')}"
    )
    rows = spark.sql(explain_sql).collect()
    plan = "\n".join(str(row["plan"]) for row in rows)
    payload.update(
        {
            "physical_plan": "== Physical Plan ==" in plan,
            "exchange": plan.count("Exchange"),
            "sort_merge_join": plan.count("SortMergeJoin"),
            "broadcast_hash_join": plan.count("BroadcastHashJoin"),
            "shuffled_hash_join": plan.count("ShuffledHashJoin"),
            "delta_scan": plan.count("DeltaScan") + plan.count("Scan delta"),
            "filter": plan.count("Filter"),
            "plan": plan[:4000],
        }
    )
except Exception as exc:  # el diagnostico nunca rompe el job
    payload["error"] = str(exc)[:300]
finally:
    for _view in (PERF_MERGE_T_VIEW, PERF_MERGE_S_VIEW):
        try:
            spark.catalog.dropTempView(_view)
        except Exception:
            pass

print("PERF_MERGE_PLAN=" + json.dumps(payload, sort_keys=True, default=str))